# 1. Data Cleaning

### 1.0 Importing the data

In [ ]:
import pandas as pd

data_path = "../data/raw_data/users.csv"
df = pd.read_csv(data_path)
df.head()

### 1.1 Handling Duplicate Values

Most duplicate data should be dropped as these will skew the data. 

Data points that need to be dropped does not necessarily need to be exact matches. For example, if a user accidentally submitted the same form twice, the "time stamp" will be different, but the rest of the information on the form will be the same.

In [ ]:
df.duplicated().sum()

As we can see here, both datasets we played with earlier has no exact matches

It is worth to analyse the data in more detail (a bit of context behind the data is needed)

For example, we can see if a unique user was mentioned twice in our dataset to see if some feature (such as the date) is the only feature that is different

In [ ]:
df1 = df.drop_duplicates(subset=["user_id"], keep="first")
print(len(df), len(df1))

Luckily, no duplicates again!

### 1.2 Inconsistent Formatting 

For ML models, "Sydney" and "sydney" mean different things, although us (humans) might write them meaning the same thing.

It's important that we have the same format for our features

In [ ]:
df.info()

In [ ]:
print(df["region"].value_counts())

### 1.3 Identifying missing values

In [ ]:
df.isnull()

In [ ]:
df.isnull().sum()

We see that the feature "region" has a few missing values in our dataset

In [ ]:
df[df["region"].isnull()].head(5)

As a general rule of thumb, if there are not too many data inputs that have a NaN value, we normally just ignore and drop them

In [ ]:
df1 = df.dropna().reset_index(drop=True)
df1.isnull().sum()

However, by doing so we lose the null data points

In [ ]:
print(f"Data points before dropping: {len(df)} \nData points after dropping: {len(df1)}")

If the data is important, or there are too many NaN inputs, we cannot just simply drop the values. 

A way to solve this is to "approximate" what the value would have been - this is called imputation. 

For numerical data, simple imputation methods are to just take the average/median of the column values that already exist - low complexity, however can lead to less accurate results. 

For categorical data, we can take the most commonly occuring category (mode), or use a "placeholder" variable value such as "unknown"

More complicated methods, such as iterative imputation, KNN imputation and ML-based imputation aims to predict the missing values with the data we have, which can provide more accurate results. However, these methods are significantly more computationally expensive. 

In [ ]:
###Here we'll impute using the mode
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
import numpy as np

imputer = SimpleImputer(missing_values=np.nan, strategy="most_frequent")
###Make sure you split your data before imputing!
X = df.drop(["churned_30d", "churned_90d"], axis=1)
###We'll only account for one target variable for now
y = df["churned_30d"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
###It's a good idea to stratify on our target variable y, as it is more likely that there are a lot more retaining customers then churning customers. 
###We can also easily check this:
print(len(df[df["churned_30d"]==0]), len(df[df["churned_30d"]==1]))

###We can now impute our data
###Make sure we fit ONLY on our training data, and then transform both our training and test set.
X_train[["region_imputed"]] = imputer.fit_transform(X_train[["region"]])
X_test[["region_imputed"]] = imputer.transform(X_test[["region"]])

In [ ]:
X_train.head()

We can check if our data is properly imputed

In [ ]:
print(len(X_train[X_train["region_imputed"]==0]))

We can now drop the old "region" feature

In [ ]:
X_train = X_train.drop(["region"], axis=1)
X_test = X_test.drop(["region"], axis=1)

Now we have done a (basic) clean of our dataset, we need to save it somewhere for later use.

In [ ]:
processed_path = "../data/processed_data"

X_train.to_csv(f"{processed_path}/X_train.csv", index=False)
X_test.to_csv(f"{processed_path}/X_test.csv", index=False)
y_train.to_csv(f"{processed_path}/y_train.csv", index=False)
y_test.to_csv(f"{processed_path}/y_test.csv", index=False)